### Multi-Agent Bidding Negotiation Training Pipeline

**Project:** Enterprise Support Router

This Colab notebook demonstrates the TRL GRPO + Unsloth training pipeline for our 4-Agent Negotiation Protocol. We leverage **Unsloth** for 4-bit Llama-3.2-1B inference, and **TRL** to optimize 4 distinct agent personas collaboratively.

In [ ]:
# 1. Install Requirements
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install openenv-py

### 2. Load the Unsloth 4-bit LLM Backbone

In [ ]:
import torch
from unsloth import FastLanguageModel

print("Loading Unsloth optimized Llama 3.2 1B Instruct model in 4-bit mode...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
model = FastLanguageModel.for_training(model)
print("Model loaded successfully!")

### 3. Initialize the Multi-Agent GRPO Pipeline
Because our OpenEnv requires a strict 3-Phase State Machine, we generate structured synthetic JSON trajectories for the Multi-Agent Personas to optimize against.

In [ ]:
import urllib.request
import os

# Download our Multi-Agent Trainer class directly from the project repository
url = "https://raw.githubusercontent.com/RavichandraNayakar/openenv-hackathon-project/main/my_env/pytorch/training/trl_multi_agent_trainer.py"
urllib.request.urlretrieve(url, "trl_multi_agent_trainer.py")

# Because we lack the full local server environment in Colab, this script imports the trainer architecture natively.
print("Downloaded PyTorch Trajectory logic for Multi-Agent negotiation.")

### 4. Run the Training Protocol
We train the 4 underlying Agent Personas sequentially using GRPO.

In [ ]:
from trl_multi_agent_trainer import MultiAgentGRPOTrainer

# Instantiate the trainer (adjust batch size to your GPU compute limits)
trainer = MultiAgentGRPOTrainer(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    learning_rate=1e-4,
    batch_size=8,
)
print("Initializing GRPO alignment...")

# Kick off the training! (Produces the training_history.json utilized by our plots)
trainer.train_all_agents(output_dir="./checkpoints_multi_agent")